In [1]:
import pickle
import os
import pandas as pd

In [2]:
res_discomat = pickle.load(open('../Matskraft_composition/code/inference/res_matskraft_composition_test.pkl', 'rb'))

In [3]:
print(res_discomat.keys())
# print(res_discomat['pred_row_col_labels'][0])
# print(res_discomat['pred_tuples'][:5])

dict_keys(['identifier', 'pred_table_type_labels', 'pred_row_col_labels', 'pred_edges', 'pred_tuples', '1_2_violations', '1_3_violations', '2_3_violations', '3_3_violations'])


In [4]:
prop_res = pickle.load(open('../Matskraft_property/data/res_dir_without_threshold/matskraft_property.pkl', 'rb'))

In [5]:
print(prop_res.keys())

dict_keys(['infer'])


In [6]:
prop_res['test'] = prop_res.pop('infer')
# print(prop_res['test'].keys())
# print(prop_res['test']['identifier'][:5])
# print(prop_res['test']['identifier'][:5])
# print(prop_res['test']['ret_comp_pred'][:5])
pred_prop_tuples_test = prop_res['test']['pred_tuples']
# print(len(pred_prop_tuples_test))
# print(pred_prop_tuples_test[0], pred_prop_tuples_test[500])

In [7]:
fpred_disco_tuples = res_discomat['pred_tuples']
pred_disco_tuples = []
for tup in fpred_disco_tuples:
    if any(tup):
        pred_disco_tuples += tup
pred_disco_tuples_test = []
for tup in pred_disco_tuples:
    #removing blank tuples
    if any(tup):
        pred_disco_tuples_test.append(tup)

In [8]:
print(len(pred_disco_tuples_test))

7161


In [9]:
print(pred_disco_tuples_test[0:5]) 
print(pred_disco_tuples_test[180:185])

[('S0022309300001034_0_0_1_0', 'Ge', 0.1, 'mol'), ('S0022309300001034_0_0_1_0', 'As', 0.15, 'mol'), ('S0022309300001034_0_0_1_0', 'Te', 0.75, 'mol'), ('S0022309300001034_0_0_2_0', 'Ge', 0.1, 'mol'), ('S0022309300001034_0_0_2_0', 'As', 0.15, 'mol')]
[('S0022309300001629_1_8_0_0', 'Al2O3', 0.05319, 'mol'), ('S0022309300001629_1_8_0_0', 'SiO2', 0.73404, 'mol'), ('S0022309300001629_1_9_0_0', 'Na2O', 0.10638, 'mol'), ('S0022309300001629_1_9_0_0', 'CaO', 0.10638, 'mol'), ('S0022309300001629_1_9_0_0', 'Al2O3', 0.10638, 'mol')]


In [10]:
#assiging orientation to props
for i in range(len(prop_res['test']['ret_comp_pred'])):
    count_col, count_row = 0, 0
    row_label = prop_res['test']['ret_comp_pred'][i]['pred_row_label']
    col_label = prop_res['test']['ret_comp_pred'][i]['pred_col_label']
    #print(row_col)
    prop_list = list(range(2,21))
    prop_res['test']['ret_comp_pred'][i]['prop_orient'] = None
    count_col = sum(1 for element in col_label if element >= 2)
    count_row = sum(1 for element in row_label if element >= 2)
    if count_col>= count_row:
        prop_res['test']['ret_comp_pred'][i]['prop_orient'] = 'col'
    else:
        prop_res['test']['ret_comp_pred'][i]['prop_orient'] = 'row'
        
#     if count_col and count_row:
#         print(prop_res['test']['identifier'][i])
#         print(prop_res['test']['ret_comp_pred'][i])
#         print()

In [11]:
def find_occ(ini_str, sub_str, occ):
    val = -1
    for i in range(0, occ):
        val = ini_str.find(sub_str, val + 1)
    return val

In [12]:
# modifying id to connect property and composition
def find_gid_prop(prop, orient):
    if orient == 'col':
        locp = find_occ(prop[0], '_', 3)
        gid_prop = prop[0][:locp]
        return gid_prop
        
    elif orient == 'row':
        locpe = find_occ(prop[0], '_', 3)
        locpi = find_occ(prop[0], '_', 2)
        gid_prop = prop[0][:locpi] + '_' + prop[0][locpe + 1]
        return gid_prop
        
    else: raise Exception(f'No orientation assigned for {pii_tid}')

In [13]:
def conn_pred_tuples_no_id(pii_tid_list):
    
    '''
    Code to connect composition and property according to orientation without any id
    only for intra table 
    
    '''
    id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set, matched_id_set = [], [], [], [], [], set(), set()
    prop_dict = {'prop_2':'d', 'prop_3':'tg', 'prop_4':'ri', 'prop_5':'abbe'}
    for pii_tid in pii_tid_list:
#         print(f'pii_tid = {pii_tid}')
        ind = prop_res['test']['identifier'].index(pii_tid)
#         pred_prop_tuples =prop_res['test']['ret_tuples_pred'][ind]
        pred_prop_tuples = result_dict[pii_tid]
        #print(len(pred_prop_tuples))
        #print(pred_prop_tuples)
        inde = res_discomat['identifier'].index(pii_tid)
        pred_comp_tuples = res_discomat['pred_tuples'][inde]
        #print(pred_comp_tuples)
        #print()

        
        prop_orient = prop_res['test']['ret_comp_pred'][ind]['prop_orient']
        for comp in pred_comp_tuples:
            gid_comp = find_gid_prop(comp, prop_orient)
            
            for prop in pred_prop_tuples:
                gid_prop = find_gid_prop(prop, prop_orient)
                if gid_comp == gid_prop:
                    matched_id_set.add(pii_tid)
                    if gid_comp not in id_list:
                        #print(f'New id == {gid_comp}')
                        id_list.append(gid_comp)
                        
                        pparts = prop[0].split('_')
                        org_id = ''
                        if len(pparts)==5:
                            org_id = pparts[-1]
                        orig_id_list.append(org_id)
                        
                        proxy_id = gid_prop.split('_')[-1]
                        if prop_orient == 'col':
                            proxy_idf = pii_tid[0] + '_' + str(pii_tid[1]) + '_' + 'R' + '_' + str(int(proxy_id) + 1) 
                        elif prop_orient == 'row':
                            proxy_idf = pii_tid[0] + '_' + str(pii_tid[1]) + '_' + 'C' + '_' + str(int(proxy_id) + 1) 
                        proxy_id_list.append(proxy_idf)
                        
                        comp_list.append([])
                        prop_list.append([])
                    insert_ind = id_list.index(gid_comp)
                    #print(insert_ind)
                    if comp[1:] not in comp_list[insert_ind]:
                        comp_list[insert_ind].append(comp[1:])
                    pr_n_list = list(prop)
                    #pr_n_list[1] = prop_dict[pr_n_list[1]]
                    prop = tuple(pr_n_list)
                    if prop[1:] not in prop_list[insert_ind]:
                        prop_list[insert_ind].append(prop[1:])
#                         print(f'{comp} ==  {prop}')
                else:
                    if gid_comp[:gid_comp.rindex('_')] != gid_prop[:gid_prop.rindex('_')]:
                        mismatch_id_set.add(pii_tid)
                    #mismatch_id_set.add(pii_tid)
            
#     for i in range(len(id_list)):
#         print(f'{id_list[i]} -> {orig_id_list} -> {proxy_id_list[i]} -> {comp_list[i]} -> {prop_list[i]}')
        #print(f'{comp_list[i]} ---> {prop_list[i]}')
        
    assert len(id_list) == len(orig_id_list) == len(proxy_id_list) == len(comp_list) == len(prop_list), 'Comp and prop length diff, not possible'
    
#     print(f'No of materials present = {len(comp_list)}')
#     print(f'No of matched ids in true matching = {len(matched_id_set)}')
    mismatch_id_set = set(pii_tid_list) - matched_id_set
#     print(f'No of mismatched ids in pred matching = {len(mismatch_id_set)}')
    #print(f'Mismatching ids == {mismatch_id_set}')
    return id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set

In [14]:
# pred_prop_tuples_test

In [15]:
### make 3 lists for pred, comp_prop, comp, prop

pred_comp_prop_pii, pred_comp_pii, pred_prop_pii = [], [], []

#make comp list for test
for tup in pred_disco_tuples_test:
    if any(tup):
#         print(tup)
        uned_pii_tid = tup[0]
        parts = uned_pii_tid.split('_')
        pii_tid = (parts[0], int(parts[1]))
        if pii_tid not in pred_comp_pii:
            pred_comp_pii.append(pii_tid)
    
#print(pred_comp_pii[:5])

#make comp list for test
for tup in pred_prop_tuples_test:
    if any(tup):
#         print(tup)
        uned_pii_tid = tup[0]
        parts = uned_pii_tid.split('_')
        pii_tid = (parts[0], int(parts[1]))
        if pii_tid not in pred_prop_pii:
            pred_prop_pii.append(pii_tid)
    
#print(pred_prop_pii[:5])

pred_comp_prop_pii = list(set(pred_comp_pii) & set(pred_prop_pii)) 
pred_only_comp_pii = list(set(pred_comp_pii) - set(pred_comp_prop_pii))
pred_only_prop_pii = list(set(pred_prop_pii) - set(pred_comp_prop_pii))
print(len(pred_comp_prop_pii), len(pred_only_comp_pii), len(pred_only_prop_pii))
print(pred_comp_prop_pii[0], pred_only_comp_pii[0], pred_only_prop_pii[0])

89 195 100
('S0022309307006813', 0) ('S0022309303007567', 1) ('S0022309304009342', 1)


In [16]:
from collections import defaultdict


result_dict = defaultdict(list)
pred_tuples = prop_res['test']['pred_tuples']
for tup in pred_tuples:
#     print(tup)
    # Split the first element of the tuple to extract the key
    key_parts = tup[0].split('_')
    key = (key_parts[0], int(key_parts[1]))
    result_dict[key].append(tup)

In [17]:
# result_dict[('S0022309303007567', 0)]

In [18]:
id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set = conn_pred_tuples_no_id(pred_comp_prop_pii)

In [19]:
def sort_comp(single_list_comp):
    '''
    for sorting  ordering of chemical components alphabetically
    '''
    single_list_comp_sorted = sorted(single_list_comp, key=lambda x: x[0])
    return single_list_comp_sorted

In [20]:
new_list_final = []
for index in range(len(id_list)):
    for prop_single in prop_list[index]:
        parts = id_list[index].split('_')
        output_string = '_'.join(parts[:2])
        my_id = orig_id_list[index]
        sorted_comp = sort_comp(comp_list[index]) # sortin existing tuples
        new_list_final.append([parts[0], [int(parts[1])+1], my_id, proxy_id_list[index], sorted_comp, prop_single])
        
print(new_list_final[:5])
print(len(new_list_final))
print(new_list_final[0])
print(len(new_list_final[0]))
start_search = len(new_list_final)

[['S0022309307006813', [1], 'A1', 'S0022309307006813_0_R_3', [('Cu', 5.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 45.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1570.0, 'K')], ['S0022309307006813', [1], 'A2', 'S0022309307006813_0_R_4', [('Cu', 15.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 35.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1560.0, 'K')], ['S0022309307006813', [1], 'A3', 'S0022309307006813_0_R_5', [('Cu', 25.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 25.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1570.0, 'K')], ['S0022309307006813', [1], 'A4', 'S0022309307006813_0_R_6', [('Cu', 35.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 15.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1470.0, 'K')], ['S0022309307006813', [1], 'A5', 'S0022309307006813_0_R_7', [('Cu', 45.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 5.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1370.0, 'K')]]
1400
['S0022309307006813', [1], 'A1', 'S0022309307006813_0_R_3', [('Cu'

In [21]:
#Appending the only comp
composition_orientation = pickle.load(open(f'composition_orientation_pred.pkl', 'rb'))
for pii_tid in pred_only_comp_pii:
    inde = res_discomat['identifier'].index(pii_tid)
    pred_comp_tuples = res_discomat['pred_tuples'][inde]
    pii_tid_w = pii_tid[0] + '_' + str(pii_tid[1])
    material = {}
    for t in pred_comp_tuples:
        id = t[0]
        if id not in material:
            material[id] = []
        material[id].append(t[1:])
#     print(material)
#     break
    for key in material.keys():
        parts = key.split('_')
        my_id = ''
        if len(parts)==6:
            my_id = parts[-1]
        output_string = '_'.join(parts[:2])
        
        if composition_orientation[(parts[0], parts[1])] == 'Row oriented':
            proxy_id = pii_tid_w + '_' + 'C' + '_' + str(int(parts[3]) + 1)
        else:
            proxy_id = pii_tid_w + '_' + 'R' + '_' + str(int(parts[2]) + 1)
        
        new_list_final.append([parts[0], [int(parts[1])+1], my_id, proxy_id, material[key], ()])

In [22]:
print(len(new_list_final))
print(new_list_final[-5:])
print(new_list_final[0])
print(new_list_final[0][0])
end_search = len(new_list_final)

2802
[['S0022309302018641', [1], 'S90C10', 'S0022309302018641_0_R_2', [('SiO2', 90.0, 'mol'), ('CaO', 10.0, 'mol'), ('SiO2', 90.62, 'wt'), ('CaO', 9.38, 'wt')], ()], ['S0022309302018641', [1], 'S80C20', 'S0022309302018641_0_R_3', [('SiO2', 80.0, 'mol'), ('CaO', 20.0, 'mol'), ('SiO2', 81.08, 'wt'), ('CaO', 18.91, 'wt')], ()], ['S0022309302018641', [1], 'S70C30', 'S0022309302018641_0_R_4', [('SiO2', 70.0, 'mol'), ('CaO', 30.0, 'mol'), ('SiO2', 71.43, 'wt'), ('CaO', 28.57, 'wt')], ()], ['S0022309302018641', [1], 'S60C40', 'S0022309302018641_0_R_5', [('SiO2', 60.0, 'mol'), ('CaO', 40.0, 'mol'), ('SiO2', 61.64, 'wt'), ('CaO', 38.36, 'wt')], ()], ['S0022309302018641', [1], 'S50C50', 'S0022309302018641_0_R_6', [('SiO2', 50.0, 'mol'), ('CaO', 50.0, 'mol'), ('SiO2', 51.72, 'wt'), ('CaO', 48.28, 'wt')], ()]]
['S0022309307006813', [1], 'A1', 'S0022309307006813_0_R_3', [('Cu', 5.0, 'mol'), ('Hf', 18.0, 'mol'), ('Ni', 45.0, 'mol'), ('Ti', 32.0, 'mol')], ('Crystallization temp', 1570.0, 'K')]
S00223

In [23]:
#Now add properties at last
for pii_tid in pred_only_prop_pii:
    inde = prop_res['test']['identifier'].index(pii_tid)
    pred_prop_tuples = result_dict[pii_tid]
    pii_tid_w = pii_tid[0] + '_' + str(pii_tid[1])
    prop_orient =prop_res['test']['ret_comp_pred'][inde]['prop_orient']
    if pii_tid == ('S0022309399003683', 1):
        print(pred_prop_tuples)
        print(prop_orient)
    #print(pred_comp_tuples)
    for tup in pred_prop_tuples:
        prop_t = (tup[1], tup[2], tup[3])
        parts = tup[0].split('_')
        my_id = ''
        if len(parts)==5:
            my_id = parts[-1]
            
        if prop_orient == 'row':
            proxy_id = pii_tid_w + '_' + 'C' + '_' + str(int(parts[3]) + 1)
        else:
            proxy_id = pii_tid_w + '_' + 'R' + '_' + str(int(parts[2]) + 1)
        
        new_list_final.append([pii_tid[0], [int(pii_tid[1])+1], my_id, proxy_id, [], prop_t])

[('S0022309399003683_1_3_1_GC1', 'Density', 2.75, 'g/cm3'), ('S0022309399003683_1_3_2_Vitreous Silica', 'Density', 2.2, 'g/cm3'), ('S0022309399003683_1_3_3_Code 1737', 'Density', 2.54, 'g/cm3'), ('S0022309399003683_1_6_1_GC1', 'Refractive index', 1.56, None), ('S0022309399003683_1_6_2_Vitreous Silica', 'Refractive index', 1.46, None), ('S0022309399003683_1_6_3_Code 1737', 'Refractive index', 1.52, None), ('S0022309399003683_1_4_1_GC1', "Young's modulus", 98.0, 'GPa'), ('S0022309399003683_1_4_2_Vitreous Silica', "Young's modulus", 72.0, 'GPa'), ('S0022309399003683_1_4_3_Code 1737', "Young's modulus", 73.0, 'GPa'), ('S0022309399003683_1_5_1_GC1', 'Vickers hardness', 650.0, 'HV'), ('S0022309399003683_1_5_2_Vitreous Silica', 'Vickers hardness', 500.0, 'HV'), ('S0022309399003683_1_5_3_Code 1737', 'Vickers hardness', 460.0, 'HV'), ('S0022309399003683_1_1_1_GC1', 'Thermal expansion coefficient', 3.6999999999999997e-06, 'degC-1'), ('S0022309399003683_1_1_2_Vitreous Silica', 'Thermal expansion 

In [24]:
print(len(new_list_final))

4577


In [25]:
deletion_indexes = set()
new_to_be_added = []
for comp_ind in range(start_search, end_search):
    for prop_ind in range(end_search, len(new_list_final)):
        if new_list_final[comp_ind][0] == new_list_final[prop_ind][0] and new_list_final[comp_ind][1] != new_list_final[prop_ind][1] and new_list_final[comp_ind][2] == new_list_final[prop_ind][2] and new_list_final[comp_ind][2]!='':
            new_to_be_added.append([new_list_final[comp_ind][0], new_list_final[comp_ind][1]+ new_list_final[prop_ind][1], new_list_final[comp_ind][2], '', new_list_final[comp_ind][4], new_list_final[prop_ind][5]])
            deletion_indexes.add(comp_ind)
            deletion_indexes.add(prop_ind)

In [26]:
inter_table_elem = len(new_to_be_added)
print(len(new_to_be_added))
print(new_to_be_added[0:5])
deletion_index = list(deletion_indexes)
deletion_index.sort()
print(len(deletion_index))
print(deletion_index[:5])
print(new_list_final[deletion_index[0]])

462
[['S0022309314005432', [1, 3], 'Gbase', '', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ('Activation energy', 656.0, 'kJ/mol')], ['S0022309314005432', [1, 3], 'Gbase', '', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ('Activation energy', 641.0, 'kJ/mol')], ['S0022309314005432', [1, 3], 'Gbase', '', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ('Activation energy', 636.0, 'kJ/mol')], ['S0022309314005432', [1, 2], 'Gbase', '', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ('Glass transition temperature', 486.0, 'degC')], ['S0022309314005432', [1, 2], 'Gbase', '', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ('Crystallization temp', 636.0, 'degC')]]
586
[1601, 1602, 1603, 1604, 1655]
['S0022309314005432', [1], 'Gbase', 'S0022309314005432_0_R_2', [('Li2O', 17.7, 'mol'), ('TiO2', 26.3, 'mol'), ('P2O5', 56.0, 'mol')], ()]


In [27]:
comp_del, prop_del, err_del = 0,0, 0
for elem in deletion_index:
    if elem<start_search:
        err_del += 1
    elif elem<end_search:
        comp_del += 1
    else:
        prop_del += 1
        
print(err_del, comp_del, prop_del)
assert err_del==0

0 128 458


In [28]:
deletion_index.sort(reverse=True)
print(deletion_index[:5])

[4550, 4549, 4548, 4547, 4546]


In [29]:
# Sort del_list in descending order to avoid index shifting issues
len_bef_del = len(new_list_final)
print(len(new_list_final))
for index in sorted(deletion_index, reverse=True):
    del new_list_final[index]
print(len(new_list_final))
del deletion_index

4577
3991


In [30]:
new_list_final_updated = new_list_final[:start_search] + new_to_be_added + new_list_final[start_search:]

In [31]:
print(len(new_list_final_updated))
del new_list_final

4453


In [32]:
print(f'Total no of composition+prop from intra table = {start_search}')
print(f'Total no of composition+prop from inter table = {inter_table_elem}')
print(f'Total no of only composition from intra table = {end_search-start_search-comp_del}')
print(f'Total no of only properties from intra table = {len_bef_del-end_search-prop_del}')
print(f'Total no of entries in the database = {len(new_list_final_updated)}')
assert len(new_list_final_updated)== start_search + inter_table_elem + (end_search-start_search-comp_del) + (len_bef_del-end_search-prop_del)

Total no of composition+prop from intra table = 1400
Total no of composition+prop from inter table = 462
Total no of only composition from intra table = 1274
Total no of only properties from intra table = 1317
Total no of entries in the database = 4453


In [33]:
deletion_indexes = []
new_to_be_added = []
for prop_ind in range(len(new_list_final_updated)-(len_bef_del-end_search-prop_del), len(new_list_final_updated)):
    for comp_ind in range(0, start_search):
        if new_list_final_updated[comp_ind][0] == new_list_final_updated[prop_ind][0] and new_list_final_updated[comp_ind][1] != new_list_final_updated[prop_ind][1] and new_list_final_updated[comp_ind][2] == new_list_final_updated[prop_ind][2] and new_list_final_updated[comp_ind][2]!='':
            new_to_be_added.append([new_list_final_updated[comp_ind][0], new_list_final_updated[comp_ind][1]+ new_list_final_updated[prop_ind][1], new_list_final_updated[comp_ind][2], '', new_list_final_updated[comp_ind][4], new_list_final_updated[prop_ind][5]])
            deletion_indexes.append(prop_ind)
            break

In [34]:
print(len(deletion_indexes))
print(len(new_to_be_added))

# Sort del_list in descending order to avoid index shifting issues
# deletion_indexes = list(deletion_indexes)
deletion_indexes.sort(reverse=True)
print(len(new_list_final_updated))

print(deletion_indexes[:5])
print(len(new_list_final_updated))

339
339
4453
[4425, 4424, 4423, 4422, 4421]
4453


In [35]:
if len(deletion_indexes)!=0:
    print('I need to write further code here')
else:
    print('Go on')

I need to write further code here


In [36]:
## Writing the further codes:

#deletion_index is reverse sorted
for index in sorted(deletion_indexes, reverse=True):
    del new_list_final_updated[index]
print(len(new_list_final_updated))
prop_del_2 = len(deletion_indexes)

del deletion_indexes

new_list_final_updated_2 = new_list_final_updated[: (start_search + inter_table_elem)] + new_to_be_added + new_list_final_updated[(start_search + inter_table_elem):]

4114


In [37]:
print(f'Total no of composition+prop from intra table = {start_search}')
print(f'Total no of composition+prop from inter table = {inter_table_elem + len(new_to_be_added)}')
print(f'Total no of only composition from intra table = {end_search-start_search-comp_del}')
print(f'Total no of only properties from intra table = {len_bef_del-end_search-prop_del-prop_del_2}')
print(f'Total no of entries in the database = {len(new_list_final_updated_2)}')
assert len(new_list_final_updated_2)== start_search + inter_table_elem + + len(new_to_be_added) + (end_search-start_search-comp_del) + (len_bef_del-end_search-prop_del-prop_del_2)

Total no of composition+prop from intra table = 1400
Total no of composition+prop from inter table = 801
Total no of only composition from intra table = 1274
Total no of only properties from intra table = 978
Total no of entries in the database = 4453


In [38]:
ch1 = start_search
ch2 = inter_table_elem + len(new_to_be_added)
ch3 = end_search-start_search-comp_del
ch4 = len_bef_del-end_search-prop_del-prop_del_2

display(pd.DataFrame(new_list_final_updated_2[ch1-1 : ch1+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2)-1 : (ch1 + ch2)+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2 + ch3)-1 : (ch1 + ch2 + ch3)+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2 + ch3 + ch4)-1 :]))

,0,1,2,3,4,5
0,S0022309304004958,[1],,S0022309304004958_0_R_11,"[(Na2O, 25.0, mol), (SiO2, 40.0, mol)]","(Melting temp, 1650.0, degC)"
1,S0022309314005432,"[1, 3]",Gbase,,"[(Li2O, 17.7, mol), (TiO2, 26.3, mol), (P2O5, ...","(Activation energy, 656.0, kJ/mol)"


,0,1,2,3,4,5
0,S0022309301006159,"[1, 2]",ZAP2,,"[(Al2O3, 25.0, mol), (P2O5, 60.0, mol), (ZnO, ...","(Young's modulus, 89.0, GPa)"
1,S0022309303007567,[2],,S0022309303007567_1_R_2,"[(P, 0.4, mol), (S, 0.6, mol)]",()


,0,1,2,3,4,5
0,S0022309313005486,[1],,S0022309313005486_0_C_3,"[(Al2O3, 2.8, mol), (K2O, 23.6, mol), (P2O5, 3...",()
1,S0022309309003883,[1],Ga0,S0022309309003883_0_R_3,[],"(Density, 2.67, g/cm3)"


,0,1,2,3,4,5
0,S0022309301007268,[4],LMBO-40%Nb2O5,S0022309301007268_3_R_12,[],"(Refractive index, 1.74, None)"


In [39]:
pickle.dump(new_list_final_updated_2, open(f'database_pred.pkl', 'wb'))

In [40]:
import pandas as pd

In [41]:
column_name = ['Article PII', 'Table No.', 'ID', 'Proxy_ID', 'Composition', 'Property']
df = pd.DataFrame(new_list_final_updated_2, columns = column_name)

In [42]:
df['Journal_Name'] = 'test_data'

In [43]:
display(pd.DataFrame(df.head()))

,Article PII,Table No.,ID,Proxy_ID,Composition,Property,Journal_Name
0,S0022309307006813,[1],A1,S0022309307006813_0_R_3,"[(Cu, 5.0, mol), (Hf, 18.0, mol), (Ni, 45.0, m...","(Crystallization temp, 1570.0, K)",test_data
1,S0022309307006813,[1],A2,S0022309307006813_0_R_4,"[(Cu, 15.0, mol), (Hf, 18.0, mol), (Ni, 35.0, ...","(Crystallization temp, 1560.0, K)",test_data
2,S0022309307006813,[1],A3,S0022309307006813_0_R_5,"[(Cu, 25.0, mol), (Hf, 18.0, mol), (Ni, 25.0, ...","(Crystallization temp, 1570.0, K)",test_data
3,S0022309307006813,[1],A4,S0022309307006813_0_R_6,"[(Cu, 35.0, mol), (Hf, 18.0, mol), (Ni, 15.0, ...","(Crystallization temp, 1470.0, K)",test_data
4,S0022309307006813,[1],A5,S0022309307006813_0_R_7,"[(Cu, 45.0, mol), (Hf, 18.0, mol), (Ni, 5.0, m...","(Crystallization temp, 1370.0, K)",test_data


In [44]:
df.to_csv(f'pred_database.csv', index=False)